In [1]:
### FOOOF FITTING

# In this code we fit fooof to all trials and all channels from a .mat file with PSD and freqs

In [2]:
# Import required code for simulating and visualizing example models

from specparam import SpectralModel
from specparam.sim import sim_power_spectrum
from specparam.sim.utils import set_random_seed
from specparam.plts.spectra import plot_spectra
from specparam.plts.annotate import plot_annotated_model
from specparam import SpectralGroupModel


import numpy as np
import scipy.io
import pandas as pd
from pathlib import Path

import os



In [ ]:
DATAIN = rf''
DATAOUT = rf''

subjects = scipy.io.loadmat(rf'')
subject_list = subjects['reftep_active_subject_list'].flatten()
subject_list = [s[0] for s in subject_list]

subjects = subject_list[0:29] # TUBINGEN ONLY


In [4]:
freq_range = [4, 40]

data_dir = DATAIN

# Storage
subj_ids = []
trial_ids = []
offset = []
exponent = []
periodic_PSD = []
aperiodic_PSD = []
MAE = []
r_sq = []

# -------------------------
# LOOP OVER SUBJECTS
# -------------------------
for subj in subjects:

    mat_file = os.path.join(data_dir, subj, f'{subj}_eeg_trials_psd.mat')

    if not os.path.exists(mat_file):
        print(f'Skipping {subj}: file not found')
        continue

    data = scipy.io.loadmat(mat_file)

    freqs = data['freqs'].flatten()
    psd = data['psd']

    freq_mask = (freqs >= freq_range[0]) & (freqs <= freq_range[1])
    model_freqs = freqs[freq_mask]

    band_mask = (model_freqs >= 8) & (model_freqs <= 13)



    # Fit FOOOF over each trial
    for trial_num in range(len(psd)):

        fm = SpectralModel(
            peak_width_limits=[2, 8],
            max_n_peaks=4,
            min_peak_height=0.1
        )

        fm.fit(freqs, psd[trial_num], freq_range)

        temp_offset = fm.get_params('aperiodic', 'offset')
        temp_exponent = fm.get_params('aperiodic', 'exponent')
        temp_MAE = fm.get_metrics('error_mae')
        temp_r_sq = fm.get_metrics('gof_rsquared')

        temp_periodic_PSD = fm.get_data('peak')
        temp_aperiodic_PSD = fm.get_data('aperiodic')

        temp_periodic_band_mean = np.mean(temp_periodic_PSD[band_mask])
        temp_aperiodic_band_mean = np.mean(temp_aperiodic_PSD[band_mask])

        subj_ids.append(subj)
        trial_ids.append(trial_num)
        offset.append(temp_offset)
        exponent.append(temp_exponent)
        MAE.append(temp_MAE)
        r_sq.append(temp_r_sq)
        periodic_PSD.append(temp_periodic_band_mean)
        aperiodic_PSD.append(temp_aperiodic_band_mean)

# -------------------------
# SAVE OUTPUT (per subject)
# -------------------------
df = pd.DataFrame({
    'subject': subj_ids,
    'trial': trial_ids,
    'periodic_PSD': periodic_PSD,
    'aperiodic_PSD': aperiodic_PSD,
    'offset': offset,
    'exponent': exponent,
    'MAE': MAE,
    'r_sq': r_sq
})

for subj in df['subject'].unique():

    subj_df = df[df['subject'] == subj]

    out_file = os.path.join(data_dir, subj, f'{subj}_fooof_results.csv')
    subj_df.to_csv(out_file, index=False)